#RS Datacatlog Uploader

Data attributes DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537155636

Interaction events DB
https://liquid-interactive.atlassian.net/wiki/spaces/MACNACA/database/4537352234

In [1]:
import dotenv
import json
dotenv.load_dotenv()

import os
import requests
import pandas as pd

RS_API_KEY = os.getenv("RUDDERSTACK_API_KEY")
RS_API_URL = "https://api.rudderstack.com"

headers = {
    "Authorization": f"Bearer {RS_API_KEY}",
    "Content-Type": "application/json",
}

In [2]:
# Read in data from CSV files

valid_types = ['object','string','number','integer','array','boolean','null']

df_properties = pd.read_csv('data/Data attributes.csv')
df_properties = df_properties[df_properties['Data attribute'].notna()] # remove rows with missing 'Data attribute' values
df_properties['Type'] = df_properties['Type'].str.lower().str.strip() # lowercase all values and remove leading/trailing whitespace 
df_properties['Type'] = df_properties['Type'].replace(['object', 'array', 'list'], 'string') # change 'object', 'array', and 'list'and  types to 'string' 


# Check properties have valid types, otherwise remove property and log error
invalid_rows = df_properties[~df_properties['Type'].isin(valid_types)]
if len(invalid_rows) > 0:
    print(f"Found {len(invalid_rows)} rows with invalid types:")
    for idx, row in invalid_rows.iterrows():
        print(f"  Row {idx}: '{row['Data attribute']}' has invalid type '{row['Type']}'")
    df_properties = df_properties[df_properties['Type'].isin(valid_types)]


df_events = pd.read_csv('data/Interaction events.csv')
#df_events.head()
df_properties.head()

,Data attribute,Description,Example value,Type,Associated with,Stream,GA4 Required?
0,interaction_type,The type of interaction a user had with the co...,"""Click""",string,component_interaction\noutlet_interaction\npro...,WS1 - Website\nWS2 - FaP\nWS2 - FaP\nWS1 - Web...,Yes
1,parent_title,The title of the container/section in which th...,"""Find support for your situation""",string,component_interaction\nsearch_result_cta_click...,WS1 - Website\nWS1 - Website\nWS2 - FaP\nWS2 -...,Yes
2,detail_title,The text associated with the specific component.,"""Read more""",string,component_interaction\nsearch_result_cta_click...,WS1 - Website\nWS1 - Website\nWS2 - FaP\nWS2 -...,Yes
3,component_id,The ID associated with the component.,"""123456""",string,component_interaction,WS1 - Website,Maybe?
4,option_selected,The text associated with the specific option a...,"""I'm looking into aged care for myself""",string,wayfinder_start\nwayfinder_next\noutlet_intera...,WS1 - Website\nWS1 - Website\nWS2 - FaP\nWS2 -...,Yes


In [ ]:
""" REMOVE, building index using properties not events
# Build an indexing list of properties within an event.  Set required to False as default.

rows = []
for idx, property in df_properties.iterrows():
    event_list = property['Associated with'].split('\n')
    for event in event_list:
        rows.append({"event": event, "property": property['Data attribute'], "required":False})
    
df_event_index = pd.DataFrame(rows)
df_event_index = df_event_index.sort_values(by=['event'])
"""

In [3]:
# Build an indexing list of properties for an event.  Set required to False as default.  
# Index should be build from Interaction Events and not Data Attributes as we can have a event with no properties in it (e.g. eol_event)

rows = []
for idx, event in df_events.iterrows():
    if pd.notna(event['Parameters']):
        
        property_list = event['Parameters'].split('\n')
        
        for property in property_list:
            rows.append({"event": event['Event name'], "property": property, "required":False})
    else:
        rows.append({"event": event['Event name'], "property": None, "required":False})
    
df_event_index = pd.DataFrame(rows)
df_event_index = df_event_index.reset_index().sort_values(by=['event', 'index']).set_index('index')  # order properties same order as in Interaction Event records

In [4]:
df_event_index

,event,property,required
index,,,
76,abandon_tool,active_question,False
77,abandon_tool,interaction_type,False
78,abandon_tool,tool,False
79,abandon_tool,context,False
229,ao_back,active_question,False
...,...,...,...
15,wayfinder_next,updated,False
16,wayfinder_next,context,False
7,wayfinder_start,context,False


In [5]:
df_required_properties = pd.DataFrame()

# If file exists, update event_index with correct required values (e.g. True = required)
try:
    df_required_properties = pd.read_csv('data/required_properties.csv')
except FileNotFoundError:
    print("File 'data/required_properties.csv' does not exist, skipping updating event_index")
    df_required_properties = pd.DataFrame()

if not df_required_properties.empty:
    # Update event_index with required properties in events 
    for idx, row in df_required_properties.iterrows():
        # use values row['event'] and row['required_property'] to find matching row in df_event_index, e.g.  df_event_index['event'] and df_event_index['property'] columns
        index_event = df_event_index[(df_event_index['event'] == row['event']) & (df_event_index['property'] == row['required_property'])]
        
        #print(index_event)
        if index_event.empty:
            print(f"Event: '{row['event']}', Property: '{row['required_property']}' not found")
        else:
            df_event_index.loc[index_event.index, 'required'] = True
            

Event: 'abandon_tool', Property: 'fake property' not found
Event: 'fake_event', Property: 'tool' not found


In [6]:
df_event_index

,event,property,required
index,,,
76,abandon_tool,active_question,False
77,abandon_tool,interaction_type,True
78,abandon_tool,tool,True
79,abandon_tool,context,False
229,ao_back,active_question,False
...,...,...,...
15,wayfinder_next,updated,False
16,wayfinder_next,context,False
7,wayfinder_start,context,False


In [ ]:
df_properties

In [14]:
# Upload properties to data catalog, if property already exists log it and skip to next record

print("Uploading properties to data catalog...")
count = 0
for idx, row in df_properties.head(2).iterrows():

    body = {
    "name": row['Data attribute'],
    "description": row['Description'],
    "type": row['Type'],
    }

    response = requests.post(f"{RS_API_URL}/v2/catalog/properties", json=body, headers=headers )

    if response.status_code == 200:
        count += 1
    else:
        print(f"Error creating property - [{response.status_code}] {response.json()['error']}")

print(f"Created {count} properties")


Uploading properties to data catalog...
Error creating property - [400] Property with name interaction_type and type string already exists
Error creating property - [400] Property with name parent_title and type string already exists
Created 0 properties


In [ ]:
# Upload events to data catalog, if event already exists log it and skip to next record
print("Uploading events to data catalog...")

count = 0
for idx, row in df_properties.head(2).iterrows():

    body = {
    "name": row['Data attribute'],
    "description": row['Description'],
    "type": row['Type'],
    }

    response = requests.post(f"{RS_API_URL}/v2/catalog/properties", json=body, headers=headers )

    if response.status_code == 200:
        count += 1
    else:
        print(f"Error creating property - [{response.status_code}] {response.json()['error']}")
